### Personal Health Data Platform - MinIO Upload

This notebook uploads customer health data to MinIO following a structured bucket layout.

**Bucket structure:**
```
health-data/
  └── customers/
        └── {customer_id}/
              ├── blood-reports/
              ├── dicom/
              ├── genomics/
              └── wearables/
```

#### 1. Install & Import

In [18]:
!pip install minio python-dotenv --quiet


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import os
import json
from pathlib import Path
from minio import Minio
from minio.error import S3Error
from dotenv import load_dotenv

load_dotenv()  # loads your .env file

True

#### 2. Connect to MinIO

In [20]:
def get_minio_client():
    """Create and return a MinIO client using .env credentials."""
    client = Minio(
        "localhost:9000",
        access_key=os.getenv("MINIO_ROOT_USER"),
        secret_key=os.getenv("MINIO_ROOT_PASSWORD"),
        secure=False  # set True if using HTTPS
    )
    return client

client = get_minio_client()
print("✅ Connected to MinIO")

✅ Connected to MinIO


#### 3. Create Bucket (if not exists)

In [21]:
BUCKET_NAME = "health-data"

def ensure_bucket(client, bucket_name):
    """Create the bucket if it doesn't already exist."""
    if not client.bucket_exists(bucket_name):
        client.make_bucket(bucket_name)
        print(f"🪣 Bucket '{bucket_name}' created.")
    else:
        print(f"✅ Bucket '{bucket_name}' already exists.")

ensure_bucket(client, BUCKET_NAME)

✅ Bucket 'health-data' already exists.


#### 4. Upload a Single File

In [22]:
def upload_file(client, local_path, minio_object_path):
    """
    Upload one file to MinIO.
    
    local_path       : path to file on your machine
    minio_object_path: destination path inside the bucket
                       e.g. 'customers/CUST_amara_patel_03630F04/blood-reports/BloodReport.pdf'
    """
    client.fput_object(BUCKET_NAME, minio_object_path, local_path)
    print(f"  ⬆️  {local_path}  →  {minio_object_path}")

#### 5. Map Local Folders → MinIO Paths

This maps each sub-folder name to its MinIO path prefix.

In [23]:
# Map local folder names to MinIO prefix names
FOLDER_MAP = {
    "BloodReports": "blood-reports",
    "DICOM":        "dicom",
    "Genomics":     "genomics",
    "Wearables":    "wearables",
}

def get_minio_prefix(customer_id, folder_name):
    """Build the MinIO object prefix for a customer's data folder."""
    minio_folder = FOLDER_MAP.get(folder_name, folder_name.lower())
    return f"customers/{customer_id}/{minio_folder}"

## 6. Upload All Files for One Customer

In [9]:
def upload_customer(client, customer_folder_path):
    """
    Upload all data for a single customer folder.
    
    customer_folder_path: Path object pointing to e.g. customers/CUST_amara_patel_03630F04
    """
    customer_id = customer_folder_path.name
    print(f"\n👤 Uploading customer: {customer_id}")

    for data_folder in customer_folder_path.iterdir():
        if not data_folder.is_dir():
            continue

        minio_prefix = get_minio_prefix(customer_id, data_folder.name)

        for file_path in data_folder.rglob("*"):
            if file_path.is_file():
                object_name = f"{minio_prefix}/{file_path.name}"
                upload_file(client, str(file_path), object_name)

    print(f"  ✅ Done: {customer_id}")

#### 7. Upload ALL Customers (Initial Full Load)

In [12]:
def upload_all_customers(client, customers_root="customers"):
    """
    Walk through every customer folder and upload all their files.
    Use this for the initial data load.
    """
    root = Path(customers_root)
    customer_folders = [f for f in root.iterdir() if f.is_dir()]
    print(f"Found {len(customer_folders)} customers to upload.")

    for folder in customer_folders:
        upload_customer(client, folder)

    print("\n🎉 All customers uploaded!")

# ▶️ Run this once for the initial load
upload_all_customers(client, customers_root="customers")

Found 5 customers to upload.

👤 Uploading customer: CUST_amara_patel_03630F04
  ⬆️  customers\CUST_amara_patel_03630F04\BloodReports\BloodReport_CUST_ama_20260511.pdf  →  customers/CUST_amara_patel_03630F04/blood-reports/BloodReport_CUST_ama_20260511.pdf
  ⬆️  customers\CUST_amara_patel_03630F04\DICOM\CT_02_CUST_ama.dcm  →  customers/CUST_amara_patel_03630F04/dicom/CT_02_CUST_ama.dcm
  ⬆️  customers\CUST_amara_patel_03630F04\DICOM\XR_01_CUST_ama.dcm  →  customers/CUST_amara_patel_03630F04/dicom/XR_01_CUST_ama.dcm
  ⬆️  customers\CUST_amara_patel_03630F04\Wearables\Wearable_CUST_ama_20260511.csv  →  customers/CUST_amara_patel_03630F04/wearables/Wearable_CUST_ama_20260511.csv
  ✅ Done: CUST_amara_patel_03630F04

👤 Uploading customer: CUST_carlos_rivera_5A81755B
  ⬆️  customers\CUST_carlos_rivera_5A81755B\BloodReports\BloodReport_CUST_car_20260511.pdf  →  customers/CUST_carlos_rivera_5A81755B/blood-reports/BloodReport_CUST_car_20260511.pdf
  ⬆️  customers\CUST_carlos_rivera_5A81755B\DICOM

---
#### 8. ➕ Add a New Customer

When you onboard a new customer, just drop their folder under `customers/` and run this cell.

In [ ]:
# ✏️ Set the new customer's folder name here
NEW_CUSTOMER_FOLDER = "customers/CUST_new_customer_XXXXXXXX"

def add_new_customer(client, customer_folder_path):
    """Upload a brand-new customer folder that didn't exist before."""
    path = Path(customer_folder_path)
    if not path.exists():
        print(f"❌ Folder not found: {customer_folder_path}")
        return
    upload_customer(client, path)

add_new_customer(client, NEW_CUSTOMER_FOLDER)

---
#### 9. 🔄 Sync New/Updated Files for an Existing Customer

This only uploads files that are **not already in MinIO**, so it's safe to re-run anytime.

In [15]:
def get_existing_objects(client, customer_id):
    """Return a set of all object names already uploaded for this customer."""
    prefix = f"customers/{customer_id}/"
    objects = client.list_objects(BUCKET_NAME, prefix=prefix, recursive=True)
    return {obj.object_name for obj in objects}


def sync_customer(client, customer_folder_path):
    """
    Upload only NEW files for an existing customer.
    Already-uploaded files are skipped (no re-upload, no overwrite).
    """
    customer_id = Path(customer_folder_path).name
    print(f"\n🔄 Syncing: {customer_id}")

    existing = get_existing_objects(client, customer_id)
    uploaded, skipped = 0, 0

    for data_folder in Path(customer_folder_path).iterdir():
        if not data_folder.is_dir():
            continue
        minio_prefix = get_minio_prefix(customer_id, data_folder.name)

        for file_path in data_folder.rglob("*"):
            if not file_path.is_file():
                continue
            object_name = f"{minio_prefix}/{file_path.name}"
            if object_name in existing:
                skipped += 1
            else:
                upload_file(client, str(file_path), object_name)
                uploaded += 1

    print(f"  ✅ {uploaded} uploaded, {skipped} already existed (skipped)")


# ✏️ Change to any customer folder you want to sync
sync_customer(client, "customers/CUST_amara_patel_03630F04")


🔄 Syncing: CUST_amara_patel_03630F04
  ✅ 0 uploaded, 4 already existed (skipped)


---
#### 10. 🔄 Sync ALL Customers at Once

Run this regularly (e.g., daily) to pick up any new files across all customers.

In [16]:
def sync_all_customers(client, customers_root="customers"):
    """Sync new files for every customer folder found locally."""
    root = Path(customers_root)
    for folder in root.iterdir():
        if folder.is_dir():
            sync_customer(client, folder)
    print("\n✅ Sync complete for all customers.")

sync_all_customers(client, customers_root="customers")


🔄 Syncing: CUST_amara_patel_03630F04
  ✅ 0 uploaded, 4 already existed (skipped)

🔄 Syncing: CUST_carlos_rivera_5A81755B
  ✅ 0 uploaded, 4 already existed (skipped)

🔄 Syncing: CUST_fatima_alsayed_8594AA83
  ✅ 0 uploaded, 4 already existed (skipped)

🔄 Syncing: CUST_john_whitfield_C4987FD5
  ✅ 0 uploaded, 4 already existed (skipped)

🔄 Syncing: CUST_mei_lin_66CBFE0F
  ✅ 0 uploaded, 4 already existed (skipped)

✅ Sync complete for all customers.


---
#### 11. 🔍 List What's Stored for a Customer (Verification)

In [17]:
def list_customer_files(client, customer_id):
    """Print all files stored in MinIO for a given customer."""
    prefix = f"customers/{customer_id}/"
    objects = client.list_objects(BUCKET_NAME, prefix=prefix, recursive=True)
    print(f"\n📂 Files in MinIO for {customer_id}:")
    for obj in objects:
        size_kb = round(obj.size / 1024, 1)
        print(f"  {obj.object_name}  ({size_kb} KB)")

list_customer_files(client, "CUST_amara_patel_03630F04")


📂 Files in MinIO for CUST_amara_patel_03630F04:
  customers/CUST_amara_patel_03630F04/blood-reports/BloodReport_CUST_ama_20260511.pdf  (3.3 KB)
  customers/CUST_amara_patel_03630F04/dicom/CT_02_CUST_ama.dcm  (8.9 KB)
  customers/CUST_amara_patel_03630F04/dicom/XR_01_CUST_ama.dcm  (8.9 KB)
  customers/CUST_amara_patel_03630F04/wearables/Wearable_CUST_ama_20260511.csv  (64.1 KB)
